# Deploy do Modelo XGBoost (pyfunc + Feature Store)

Este notebook faz o deploy do **modelo XGBoost** para servico em **tempo real** via Feature Store:

**Workflow:**
1. Busca o melhor run XGBoost do MLflow (maior PR-AUC no validacao)
2. Carrega o modelo XGBoost + recria o preprocessor (ColumnTransformer)
3. Testa predicoes no conjunto de teste para validar
4. Registra no Unity Catalog via `FeatureEngineeringClient.log_model()` com **FeatureLookup**
5. O serving endpoint recebe apenas `TransactionID` + `TransactionDT` — o Feature Store busca as 150+ features automaticamente
6. O pyfunc preprocessa (encoder + scaler + imputer), prediz e aplica threshold

**Arquitetura real-time:**
```
Cliente envia TransactionID + TransactionDT
  → Feature Store lookup (150+ features RAW)
  → pyfunc: preprocess + XGBoost + threshold
  → {probability_fraud, is_fraud}
```

**Unity Catalog:** `fraud_detection_dev.models.xgboost_fraud_model`

In [0]:
%pip install xgboost databricks-feature-engineering -q

In [0]:
# Busca o melhor modelo XGBoost no MLflow baseado em PR-AUC

import sys
import mlflow
from mlflow import MlflowClient
import pandas as pd
import numpy as np
import json
import logging

# Carrega configuração central do YAML
sys.path.insert(0, "/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/src")
from config import load_config
cfg = load_config()

# Extrai constantes do config
FEATURE_TABLE = cfg["feature_store"]["table"]
HPO_EXPERIMENT_NAME = cfg["experiments"]["hpo"]
DEPLOY_EXPERIMENT_NAME = cfg["experiments"]["deploy"]
OPTIMAL_THRESHOLD = cfg["model"]["optimal_threshold"]
SEED = cfg["model"]["seed"]
UC_MODEL_NAME = cfg["model"]["registered_name"]  # Unity Catalog (from config)

# Suprime warnings
logging.getLogger("mlflow.tracking.context.registry").disabled = True

print("=" * 80)
print("BUSCA DO MELHOR MODELO XGBOOST NO MLFLOW")
print("=" * 80)

# Busca o experimento de HPO do notebook 03
experiment = mlflow.get_experiment_by_name(HPO_EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experimento não encontrado: {HPO_EXPERIMENT_NAME}. Execute o notebook 03 primeiro.")

client = MlflowClient()

# Busca todos os runs com PR-AUC > 0, ordenados por PR-AUC decrescente
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="metrics.pr_auc > 0",
    order_by=["metrics.pr_auc DESC"],
    max_results=1  # Só precisa do melhor
)

if not runs:
    raise ValueError(
        f"Nenhum run encontrado no experimento {HPO_EXPERIMENT_NAME}.\n"
        "Execute o notebook 03_MODEL_TRAINING primeiro para treinar o modelo XGBoost."
    )

best_run = runs[0]
best_run_id = best_run.info.run_id
best_pr_auc = best_run.data.metrics.get("pr_auc", 0.0)

# Extrai métricas principais do melhor run
metrics_to_show = [
    "pr_auc", "roc_auc", "precision", "recall", "f1_score",
    "brier_score", "ks_statistic"
]
best_metrics = {
    k: best_run.data.metrics.get(k, None)
    for k in metrics_to_show
    if k in best_run.data.metrics
}

print(f"\nMelhor Run ID: {best_run_id}")
print(f"\nMétricas (conjunto de validação):")
for k, v in best_metrics.items():
    if v is not None:
        print(f"  {k:<20}: {v:.4f}")

print(f"\nThreshold ótimo: {OPTIMAL_THRESHOLD}")
print(f"Feature Table:    {FEATURE_TABLE}")
print(f"Modelo UC:        {UC_MODEL_NAME}")
print("=" * 80)

In [0]:
# Carrega conjunto de teste para validação do modelo

print("Carregando dados de teste...")

# Lê a tabela da feature store (Spark) e converte para pandas
df_spark = spark.read.table(FEATURE_TABLE)
df_test_spark = df_spark.filter("split = 'test'")

print(f"Total de linhas no teste: {df_test_spark.count():,}")

# Converte para pandas (o modelo XGBoost espera pandas DataFrame)
df_test = df_test_spark.toPandas()

# Separa features e target
X_test = df_test.drop(columns=["TransactionID", "target", "split"])
y_test = df_test["target"]

print(f"\nConjunto de teste (pandas):")
print(f"  Linhas:        {len(X_test):,}")
print(f"  Features:      {len(X_test.columns)}")
print(f"  Target:        {y_test.sum():,} fraudes ({y_test.mean()*100:.2f}%)")
print(f"\nPrimeiras features: {list(X_test.columns[:5])}...")

In [0]:
# Carrega modelo XGBoost + recria preprocessor

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

print("Buscando o run pai com o modelo XGBoost no MLflow...")

# O best_run_id da célula 2 aponta para um trial filho do Optuna (sem artifact de modelo).
# O modelo foi logado via mlflow.xgboost.log_model() apenas no run pai "XGBoost_Optuna_HPO".
parent_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'XGBoost_Optuna_HPO'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

if not parent_runs:
    raise ValueError(
        "Run pai 'XGBoost_Optuna_HPO' não encontrado no experimento "
        f"{HPO_EXPERIMENT_NAME}. Execute o notebook 03_MODEL_TRAINING primeiro."
    )

model_run = parent_runs[0]
model_run_id = model_run.info.run_id

print(f"✓ Run pai encontrado: {model_run_id}")

# Carrega o modelo XGBoost do MLflow
model_uri = f"runs:/{model_run_id}/model"
xgb_model = mlflow.xgboost.load_model(model_uri)

print(f"✓ Modelo XGBoost carregado: {type(xgb_model).__name__}")

# Recria o preprocessor (mesmo código do notebook 03)
# Usa exclude_cols do config central + split (presente na feature store)
exclude_cols = set(cfg["preprocessing"]["exclude_cols"]) | {"split"}
feature_names = [c for c in X_test.columns if c not in exclude_cols]
cat_cols = [c for c in feature_names if X_test[c].dtype == "object"]
num_cols = [c for c in feature_names if c not in cat_cols]

print(f"\nPreprocessor:")
print(f"  Features categóricas: {len(cat_cols)}")
print(f"  Features numéricas:   {len(num_cols)}")
print(f"  Total:                {len(feature_names)}")

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
])

# Carrega dados de TREINO para ajustar o preprocessor (evita data leakage)
print("\nCarregando dados de treino para ajustar o preprocessor...")
df_train = spark.read.table(FEATURE_TABLE).filter("split = 'train'").toPandas()

print(f"Ajustando preprocessor no conjunto de treino ({len(df_train):,} linhas)...")
preprocessor.fit(df_train[feature_names])

In [0]:
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

print("Gerando predicoes no conjunto de teste...")

# Preprocessa os dados brutos e prediz com o XGBoost
X_test_features = X_test[feature_names]
X_test_processed = preprocessor.transform(X_test_features)
y_proba = xgb_model.predict_proba(X_test_processed)
y_proba_fraud = y_proba[:, 1]

y_pred = (y_proba_fraud >= OPTIMAL_THRESHOLD).astype(int)

test_metrics = {
    "pr_auc": average_precision_score(y_test, y_proba_fraud),
    "roc_auc": roc_auc_score(y_test, y_proba_fraud),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("\n" + "=" * 80)
print("PERFORMANCE NO CONJUNTO DE TESTE")
print("=" * 80)
print(f"PR-AUC:      {test_metrics['pr_auc']:.4f}")
print(f"ROC-AUC:     {test_metrics['roc_auc']:.4f}")
print(f"Precision:   {test_metrics['precision']:.4f}")
print(f"Recall:      {test_metrics['recall']:.4f}")
print(f"F1-Score:    {test_metrics['f1_score']:.4f}")
print(f"\nThreshold usado: {OPTIMAL_THRESHOLD}")
print(f"\nMatriz de Confusao:")
print(f"  TN: {tn:>6,}  |  FP: {fp:>6,}")
print(f"  FN: {fn:>6,}  |  TP: {tp:>6,}")
print(f"\nTaxa de fraudes detectadas: {tp / (tp + fn):.2%}")
print(f"Taxa de falsos positivos:   {fp / (fp + tn):.2%}")
print("=" * 80)

In [0]:
# Em producao real-time, o cliente envia apenas TransactionID + TransactionDT
# O Feature Store busca as 150+ features automaticamente da tabela serving compacta
SERVING_TABLE = "fraud_detection_dev.gold.fraud_features_serving"

scoring_sample = (
    spark.read.table(SERVING_TABLE)
    .select("TransactionID", "TransactionDT")
    .limit(5)
)
print("Input do serving (apenas chaves de lookup):")
display(scoring_sample)

In [0]:
# Testa com fe.score_batch — envia apenas TransactionID + TransactionDT
# O Feature Store busca as 150+ features automaticamente via FeatureLookup
from databricks.feature_engineering import FeatureEngineeringClient

SERVING_TABLE = "fraud_detection_dev.gold.fraud_features_serving"
fe = FeatureEngineeringClient()
model_uri_champion = f"models:/{UC_MODEL_NAME}@champion"

# Usa TransactionIDs da serving table (split=val) — o FeatureLookup só encontra estas chaves
scoring_df = (
    spark.read.table(SERVING_TABLE)
    .select("TransactionID", "TransactionDT")
    .limit(1000)
)

# fe.score_batch retorna a probabilidade na coluna "prediction"
predictions = fe.score_batch(model_uri=model_uri_champion, df=scoring_df)
pred_pd = predictions.select("TransactionID", "prediction").toPandas()

# Aplica threshold 
pred_pd["probability_fraud"] = pred_pd["prediction"]
pred_pd["is_fraud"] = (pred_pd["probability_fraud"] >= OPTIMAL_THRESHOLD).astype(int)

# Compara com labels reais (target está na Gold original, não na serving)
labels_pd = (
    spark.read.table(FEATURE_TABLE)
    .filter("split = 'val'")
    .select("TransactionID", "target")
    .limit(1000)
    .toPandas()
)
y_sample = labels_pd.set_index("TransactionID").loc[pred_pd["TransactionID"]]["target"].values

# Metricas de avaliacao (PR-AUC, KS, ROC-AUC, Brier)
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from scipy.stats import ks_2samp

pr_auc = average_precision_score(y_sample, pred_pd["probability_fraud"])
roc_auc = roc_auc_score(y_sample, pred_pd["probability_fraud"])
brier = brier_score_loss(y_sample, pred_pd["probability_fraud"])
ks_stat = ks_2samp(
    pred_pd.loc[y_sample == 1, "probability_fraud"],
    pred_pd.loc[y_sample == 0, "probability_fraud"],
).statistic

print("=" * 80)
print(f"Modelo: {model_uri_champion}")
print(f"Input: apenas TransactionID + TransactionDT ({len(pred_pd):,} amostras)")
print(f"  PR-AUC:    {pr_auc:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")
print(f"  KS:        {ks_stat:.4f}")
print(f"  Brier:     {brier:.4f}")
print("\nFeature Store lookup funcionando — modelo recebe apenas TransactionID.")
print("=" * 80)

## Resumo do Deploy

O modelo **XGBoost** foi deployado no Unity Catalog como **pyfunc + Feature Store**:

**Modelo:**
* Nome UC: `fraud_detection_dev.models.xgboost_fraud_model`
* Alias: `@champion`
* Tipo: pyfunc (PythonModel) + FeatureLookup
* Algoritmo: XGBoost + sklearn ColumnTransformer
* Threshold: 0.83 (aplicado dentro do pyfunc)

**Serving real-time:**
```python
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()

# Cliente envia apenas TransactionID + TransactionDT
scoring_df = spark.createDataFrame([(12345, 99999)], ["TransactionID", "TransactionDT"])

# Feature Store busca as 150+ features automaticamente
predictions = fe.score_batch(
    model_uri="models:/fraud_detection_dev.models.xgboost_fraud_model@champion",
    df=scoring_df,
)
# predictions: TransactionID, TransactionDT, probability_fraud, is_fraud
```

In [0]:
# Deploy do modelo pyfunc com FeatureLookup apontando para a tabela serving compacta
# A tabela fraud_features_serving tem apenas split='val' (114k linhas, ~134MB)
# em vez da Gold completa (590k linhas, ~574MB) — cabe no online store CU_8

import mlflow
import mlflow.pyfunc
import pickle
import json
import tempfile
import os
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# Reutiliza variaveis das celulas anteriores:
#   xgb_model, preprocessor, feature_names, test_metrics, model_run_id, best_run, cfg, client

SERVING_TABLE = "fraud_detection_dev.gold.fraud_features_serving"
UC_MODEL_NAME = cfg["model"]["registered_name"]
DEPLOY_EXPERIMENT_NAME = cfg["experiments"]["deploy"]
OPTIMAL_THRESHOLD = cfg["model"]["optimal_threshold"]

mlflow.set_experiment(DEPLOY_EXPERIMENT_NAME)
mlflow.set_registry_uri("databricks-uc")
_uc_parts = UC_MODEL_NAME.split(".")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {_uc_parts[0]}.{_uc_parts[1]}")

fe = FeatureEngineeringClient()


class FraudDetectionModel(mlflow.pyfunc.PythonModel):
    """Pyfunc: recebe features RAW do Feature Store, preprocessa, prediz e aplica threshold."""

    def load_context(self, context):
        import pickle
        import json

        with open(context.artifacts["preprocessor"], "rb") as f:
            self.preprocessor = pickle.load(f)

        with open(context.artifacts["model"], "rb") as f:
            self.model = pickle.load(f)

        with open(context.artifacts["config"], "r") as f:
            config = json.load(f)
            self.threshold = config["threshold"]
            self.feature_names = config["feature_names"]

    def predict(self, context, model_input, params=None):
        # model_input vem do Feature Store lookup (features RAW)
        X = model_input[self.feature_names]
        X_processed = self.preprocessor.transform(X)
        proba = self.model.predict_proba(X_processed)[:, 1]

        return pd.DataFrame({
            "probability_fraud": proba,
            "is_fraud": (proba >= self.threshold).astype(int),
        })

# 1. FeatureLookup apontando para a tabela serving compacta (em vez da Gold completa)
feature_lookups_serving = [
    FeatureLookup(
        table_name=SERVING_TABLE,
        lookup_key="TransactionID",
        timestamp_lookup_key="TransactionDT",
        feature_names=feature_names,
    )
]

# 2. Labels DataFrame: TransactionID + TransactionDT + target
#    target NAO existe na serving table, então vem da Gold original
labels_df = spark.read.table(FEATURE_TABLE).select("TransactionID", "TransactionDT", "target")

# 3. Training set: labels da Gold + features da serving table via FeatureLookup
serving_training_set = fe.create_training_set(
    df=labels_df,
    feature_lookups=feature_lookups_serving,
    label="target",
)

print(f"FeatureLookup -> {SERVING_TABLE}")
print(f"Features: {len(feature_names)}")
print(f"Labels da: {FEATURE_TABLE} (apenas chaves + target)")

# 4. Re-loga o modelo pyfunc com o novo FeatureSpec
with tempfile.TemporaryDirectory() as tmp:
    preprocessor_path = os.path.join(tmp, "preprocessor.pkl")
    with open(preprocessor_path, "wb") as f:
        pickle.dump(preprocessor, f)

    model_path = os.path.join(tmp, "model.pkl")
    with open(model_path, "wb") as f:
        pickle.dump(xgb_model, f)

    config_path = os.path.join(tmp, "config.json")
    with open(config_path, "w") as f:
        json.dump({
            "threshold": OPTIMAL_THRESHOLD,
            "feature_names": feature_names,
        }, f)

    # Infer signature
    sample_input = X_test[feature_names].head(100)
    sample_proba = xgb_model.predict_proba(preprocessor.transform(sample_input))[:, 1]
    sample_output = pd.DataFrame({
        "probability_fraud": sample_proba,
        "is_fraud": (sample_proba >= OPTIMAL_THRESHOLD).astype(int),
    })
    signature = infer_signature(sample_input, sample_output)

    with mlflow.start_run(run_name="fraud_detection_pyfunc_serving_v2") as run:
        mlflow.log_params(best_run.data.params)
        mlflow.log_param("optimal_threshold", OPTIMAL_THRESHOLD)
        mlflow.log_param("feature_table", SERVING_TABLE)
        mlflow.log_param("training_run_id", model_run_id)
        mlflow.log_param("model_type", "pyfunc")
        mlflow.log_param("serving_mode", "feature_store_lookup")
        mlflow.log_param("serving_table", "compact_val_only")

        for metric_name, metric_value in test_metrics.items():
            mlflow.log_metric(f"test_{metric_name}", metric_value)

        # Log com FeatureEngineeringClient — empaquota FeatureSpec (serving table) junto ao pyfunc
        fe.log_model(
            model=FraudDetectionModel(),
            flavor=mlflow.pyfunc,
            artifact_path="model",
            training_set=serving_training_set,
            artifacts={
                "preprocessor": preprocessor_path,
                "model": model_path,
                "config": config_path,
            },
            pip_requirements=["xgboost", "scikit-learn", "pandas", "numpy"],
            registered_model_name=UC_MODEL_NAME,
        )

        deploy_run_id = run.info.run_id

# 5. Configura tags e alias @champion na nova versao
client = MlflowClient(registry_uri="databricks-uc")
all_versions = client.search_model_versions(f"name='{UC_MODEL_NAME}'")
latest_version = max(all_versions, key=lambda v: int(v.version))
model_version = latest_version.version

client.set_model_version_tag(UC_MODEL_NAME, model_version, "algorithm", "XGBoost")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "model_type", "pyfunc")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "serving_mode", "feature_store_lookup")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "serving_table", "fraud_features_serving")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "optimal_threshold", str(OPTIMAL_THRESHOLD))
client.set_model_version_tag(UC_MODEL_NAME, model_version, "test_pr_auc", f"{test_metrics['pr_auc']:.4f}")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "test_f1_score", f"{test_metrics['f1_score']:.4f}")
client.set_model_version_tag(UC_MODEL_NAME, model_version, "training_run_id", str(model_run_id))

client.set_registered_model_alias(UC_MODEL_NAME, "champion", int(model_version))

model_uri = f"models:/{UC_MODEL_NAME}@champion"

print("=" * 80)
print(f"Modelo re-logado com FeatureLookup -> {SERVING_TABLE}")
print(f"Modelo: {UC_MODEL_NAME}")
print(f"Versao: {model_version}")
print(f"Alias:  @champion")
print(f"URI:    {model_uri}")
print(f"Deploy run ID: {deploy_run_id}")
print("\nMetricas de teste:")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")
print("=" * 80)